In [ ]:
!pip install nipype nilearn matplotlib pandas

In [ ]:
!pip install scipy numpy scikit-learn

In [ ]:
import os
from nipype import Node, Workflow
from nipype.interfaces import fsl
from nilearn import image, masking, datasets, plotting
from nilearn.connectome import ConnectivityMeasure
import pandas as pd
import matplotlib.pyplot as plt

# Define local directory path
data_dir = 'data/ds000051'
t1_file = os.path.join(data_dir, 'sub-01/anat/sub-01_run-01_T1w.nii.gz')
bold_file = os.path.join(data_dir, 'sub-01/func/sub-01_task-abstractconcretejudgment_run-01_bold.nii.gz')
working_dir = os.path.join(data_dir, 'working_dir')

# Ensure working directory exists
os.makedirs(working_dir, exist_ok=True)

# Define paths for output files
bold_mc_file = os.path.join(working_dir, 'bold_mc.nii.gz')
bold_coreg_file = os.path.join(working_dir, 'bold_coreg.nii.gz')

# Define nodes for preprocessing
motion_correct = Node(fsl.MCFLIRT(in_file=bold_file, out_file=bold_mc_file), name='motion_correct')
coregister = Node(fsl.FLIRT(in_file=bold_mc_file, reference=t1_file, out_file=bold_coreg_file), name='coregister')

# Set up workflow
workflow = Workflow(name='preprocessing')
workflow.base_dir = working_dir

# Add nodes to the workflow
workflow.add_nodes([motion_correct, coregister])

# Connect nodes
workflow.connect([
    (motion_correct, coregister, [('out_file', 'in_file')])
])

# Check if files are created
def check_file_exists(filepath):
    if os.path.exists(filepath):
        print(f"File exists: {filepath}")
    else:
        print(f"File does not exist: {filepath}")

# Run the workflow
print("Running preprocessing workflow...")
workflow.run()

# Check if motion-corrected file is created
check_file_exists(bold_mc_file)

# Check if coregistered file is created
check_file_exists(bold_coreg_file)

# Load the preprocessed fMRI data
func_img = image.load_img(bold_coreg_file)

# Load Harvard-Oxford atlas
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-1mm')
atlas_filename = atlas['filename']
atlas_labels = atlas['labels']

# Apply masking
masker = masking.NiftiLabelsMasker(labels_img=atlas_filename, standardize=True, detrend=True)
time_series = masker.fit_transform(func_img)

# Compute connectivity matrix
connectivity_measure = ConnectivityMeasure(kind='correlation')
connectivity_matrix = connectivity_measure.fit_transform([time_series])[0]

# Convert to DataFrame for easier viewing
connectivity_df = pd.DataFrame(connectivity_matrix, index=atlas_labels, columns=atlas_labels)
print(connectivity_df)

# Plot connectivity matrix
plt.figure(figsize=(10, 8))
plotting.plot_matrix(connectivity_matrix, figure=(10, 8), labels=atlas_labels, vmax=1, vmin=-1, reorder=True, title='ROI-to-ROI Connectivity Matrix')
plt.show()
